# Calculating semantic tag statistics for verbs

This notebook calculate various statistical information about what type of adverbs different verbs take and puts the results into database tables.

For that purpose we go through the following steps:
1. Count for each verb how many adverbs were annotated as an user specified tag, other tags, semantically annotated at all or recieved no semantic tag
2. Calculate percentages for how many of the verbs had adverbial dependents that are an user specified tag, other tags, annotated and not_annotated based on the counts from step 1.
3. Calculate proportion of location/other and annotated/not_annotated with binary logarithms. These will later be used to make graph showing if a verb's adverbial dependents are more likely some user specified tags or other tags and how many of the words are annotated at all
5. Find how many unique adverbs each verb takes, calculating its binary algorithm. Used later as a hoverplot's x-axis to show how much the tag proportions can be trusted.

In [22]:
import sqlite3
import pandas as pd
import numpy as np

In [23]:
# database file path
filename = "C:\\Users\\kertu.saul\\OneDrive - Eesti Keele Instituut\\Dokumendid\\doktoritoo\\ressurssid\\rektsioonid\\katrin\\andmebaasifailid\\v33_koondkorpus_transaktsioonid.db"

# connecting with database
conn = sqlite3.connect(filename)
cursor = conn.cursor()

## 1. Count semantic tags
Create a table with pure counts where each verb has how many of that verb's adverbial dependents have user defined semantic tags, other tags, are annotated, aren't annotated and how many times the verb took an advcerbial dependent. The tag 'general' for üldlaiendid is left out of the equations since they are never verb-specific.

The user has to insert the semantic tags they want counts of while calling the function count_table

In [24]:
#create a table that counts tags for verbs
#tag1 = first tag to count
#tag2 = tag you want to count together with the second tag
def count_table_adv(file, tag1, tag2 = ''):
    #Connect to database
    conn = sqlite3.connect(file)
    cursor = conn.cursor()

    #define tag name in database table
    if tag2 == '':
        tag = tag1
    else:
        tag = tag1 + '_' + tag2
    
    #delete table if it exists
    cursor.execute("DROP TABLE IF EXISTS verb_adv_counts_"+tag)

    # Step 1: Create the new counts table 
    cursor.execute("""
            CREATE TABLE IF NOT EXISTS verb_adv_counts_"""+tag+""" (
            verb TEXT,
            verb_compound TEXT,
            my_tag INT,
            other_tags INT,
            annotated INT,
            not_annotated INT,
            verb_adv_count INT
        )
    """)

    # Step 2: Aggregate counts
    cursor.execute(
    f"""INSERT INTO verb_adv_counts_{tag}
        (verb, verb_compound, my_tag, other_tags, annotated, not_annotated, verb_adv_count)
        SELECT 
            verb, 
            verb_compound, 
            COUNT(CASE WHEN ekilex_tag = ? OR ekilex_tag = ? THEN 1 END) AS my_tag,
            COUNT(CASE WHEN ekilex_tag IS NOT NULL AND ekilex_tag != 'general' AND ekilex_tag != ? AND ekilex_tag != ? THEN 1 END) AS other_tags,
            COUNT(CASE WHEN ekilex_tag IS NOT NULL AND ekilex_tag != 'general' THEN 1 END) AS annotated,
            COUNT(CASE WHEN ekilex_tag IS NULL OR ekilex_tag = 'general' THEN 1 END) AS not_annotated,
            COUNT(*) AS verb_adv_count
        FROM advmod
        GROUP BY verb, verb_compound
    """, (tag1, tag2, tag1, tag2)
    )

    # Commit and close
    conn.commit()
    conn.close()

In [6]:
#create count table for location vs other tags
count_table_adv(filename, 'location')

In [25]:
#create count table for manner vs other tags
count_table_adv(filename, 'manner')

In [36]:
#create count table for time vs other tags
count_table_adv(filename, 'time')

In [47]:
#create count table for amount vs other tags
count_table_adv(filename, 'amount')

In [56]:
#create count table for state vs other tags
count_table_adv(filename, 'state')

## 2. Calculate percentages
Uses the counts from the previous table to calculate percentages of each class for every verb+case pair
* my_tag_pr = what percentage of annotated words had an user specified tag. User can specify multiple tags
* other_tags_pr: what percentage of annotated words didn't have those tags
* annotated_pr: what percentage of words were annotated
* not_annotated_pr: what percentage of words were not annotated

Users have to define database table name

Results are put into a new dataframe with a verb, verb compound and the percentages specified above

In [26]:
def semtype_percentages(file, table_name):
    #connect to database
    conn = sqlite3.connect(file)
    cursor = conn.cursor()
    
    #read counts into dataframe
    df = pd.read_sql("SELECT * FROM " + table_name, conn)

    #only take rows that have more than 4 examples
    df = df.loc[df['verb_adv_count'] > 4]

    #remove verb+case pairs that have no annotated dependents
    df = df.loc[df['annotated'] != 0]

    # Calculate percentages
    df["my_tag_pr"] = df["my_tag"] / df["annotated"]
    df["other_tags_pr"] = df["other_tags"] / df["annotated"]
    df["annotated_pr"] = df["annotated"] / df["verb_adv_count"]
    df["not_annotated_pr"] = df["not_annotated"] / df["verb_adv_count"]

    #create a new dataframe with only percentages
    df_pr = df[['verb', 'verb_compound', "my_tag_pr", 'other_tags_pr', 'annotated_pr', 'not_annotated_pr', 'verb_adv_count']].copy()

    # Commit and close
    conn.commit()
    conn.close()

    return df_pr

In [10]:
#percentage dataframe for location
df_pr_loc = semtype_percentages(filename, 'verb_adv_counts_location')
#example
df_pr_loc.loc[df_pr_loc['verb'] == 'käima']

,verb,verb_compound,my_tag_pr,other_tags_pr,annotated_pr,not_annotated_pr,verb_adv_count
10086,käima,,0.231955,0.768045,0.546578,0.453422,80832
10089,käima,all,0.363636,0.636364,0.647059,0.352941,17
10092,käima,alla,0.404444,0.595556,0.520833,0.479167,432
10094,käima,alles,0.315789,0.684211,0.593750,0.406250,32
10098,käima,edasi,0.196429,0.803571,0.383562,0.616438,292
10099,käima,"edasi, tagasi",0.285714,0.714286,0.777778,0.222222,9
10100,käima,edasi-tagasi,0.000000,1.000000,0.600000,0.400000,15
10101,käima,ees,0.263158,0.736842,0.558824,0.441176,34
10103,käima,ette,0.333333,0.666667,0.480000,0.520000,25
10106,käima,ilma,0.333333,0.666667,0.600000,0.400000,5


In [27]:
#percentage dataframe for manner
df_pr_manner = semtype_percentages(filename, 'verb_adv_counts_manner')

In [37]:
#percentage dataframe for time
df_pr_time = semtype_percentages(filename, 'verb_adv_counts_time')

In [48]:
#percentage dataframe for amount
df_pr_amount = semtype_percentages(filename, 'verb_adv_counts_amount')

In [57]:
#percentage dataframe for state
df_pr_state = semtype_percentages(filename, 'verb_adv_counts_state')

## 4. Calculate proportions
To better illustrate whether a verb prefers adverbials with the user specified tag (ie location) or all the other tags (ie time+state+manner+amount), we calculate pointwise mutual information by dividing an user specified tag(s) percentage by every other_tags percentage and taking a binary logarithm of it. 

This section:
* Calculates PMI for tag vs other_tags and annotated vs not_annotated per verb
* Adds logarithms to dataframe and
* Transforms the dataframe into a database table

In [28]:
def semtype_log(file, df_pr, table_name):

    #replace zeros with 0,001 to avoid taking log from zero
    #not replacing zero with a VERY small number like 1e-10 to avoid graph stretching out
    df_filt = df_pr.replace(0.0, 0.001)

    #calculate binary logarithm for specific tag(s) vs other tags
    df_filt["log2_tag"] = np.log2((df_filt["my_tag_pr"]) / (df_filt["other_tags_pr"]))

    #calculate binary logarithm for annotated/not_annotated
    df_filt["log2_annotation"] = np.log2((df_filt["annotated_pr"]) / (df_filt["not_annotated_pr"]))

    #create a new dataframe with only the proportions
    df_log2 = df_filt[['verb', 'verb_compound', 'log2_tag', 'log2_annotation', 'verb_adv_count']].copy()
    
    #connect to database
    conn = sqlite3.connect(file)
    cursor = conn.cursor()

    #drop table if it exists
    cursor.execute("DROP TABLE IF EXISTS " + table_name)

    # Step 1: Create the new results table 
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS """ +  table_name  + """ (
            verb TEXT,
            verb_compound TEXT,
            log2_tag REAL,
            log2_annotation REAL,
            verb_adv_count INT
        )
    """)

    # Step 2: Insert percentages from the dataframe into the database table
    df_log2.to_sql(table_name, conn, if_exists="replace", index=False)

    # Commit and close
    conn.commit()
    conn.close()

In [12]:
#create log table for locations
semtype_log(filename, df_pr_loc, 'verb_adv_log_location')

In [29]:
semtype_log(filename, df_pr_manner, 'verb_adv_log_manner')

In [38]:
semtype_log(filename, df_pr_time, 'verb_adv_log_time')

In [49]:
semtype_log(filename, df_pr_amount, 'verb_adv_log_amount')

In [58]:
semtype_log(filename, df_pr_state, 'verb_adv_log_state')

## 5. Find how many different words each verb+case pair has

This is to show how trustworthy the tag proportions are for a verb. The more different words a verb's adverbial dependents are, the more trustworthy the results are, because the sample was larger

#### Find unique lemma counts for verb+case pairs, calculate binary logarithm for them

In [59]:
#Connect to database
conn = sqlite3.connect(filename)
cursor = conn.cursor()

# Aggregate counts
query = """
    SELECT 
        verb, 
        verb_compound,
        COUNT(DISTINCT lemma) AS unique_lemmas
    FROM advmod
    WHERE ekilex_tag is not null and ekilex_tag != 'general'
    GROUP BY verb, verb_compound
"""

# Make into dataframe to calculate log2
df_unique_lemmas = pd.read_sql(query, conn)

#calculate binary logarithm for unique lemmas
df_unique_lemmas["log2_unique_lemmas"] = np.log2(df_unique_lemmas["unique_lemmas"]) 

#### Create a new database table for the unique lemma counts and logarithms

In [60]:
#delete table if it exists
cursor.execute("DROP TABLE IF EXISTS verb_adv_unique_lemmas")

# Step 1: Create the new counts table 
cursor.execute("""
    CREATE TABLE IF NOT EXISTS verb_adv_unique_lemmas (
        verb TEXT,
        verb_compound TEXT,
        unique_lemmas INT,
        log2_unique_lemmas REAL
    )
""")

df_unique_lemmas.to_sql("verb_adv_unique_lemmas", conn, if_exists="replace", index=False)

# Commit and close
conn.commit()
conn.close()

#### Add unique lemma columns to log statistics tables to ease hoverplot creation

In [61]:
def new_columns(table_name):
    #Connect to database
    conn = sqlite3.connect(filename)
    cursor = conn.cursor()

    cursor.execute("""
        ALTER TABLE """ + table_name + """
        ADD COLUMN unique_lemmas INTEGER;
    """)

    cursor.execute("""
        ALTER TABLE """ + table_name + """
        ADD COLUMN log2_unique_lemmas REAL;
    """)

In [ ]:
new_columns('verb_adv_log_location')

In [33]:
new_columns('verb_adv_log_manner')

In [42]:
new_columns('verb_adv_log_time')

In [53]:
new_columns('verb_adv_log_amount')

In [62]:
new_columns('verb_adv_log_state')

#### Add unique lemma count data to log statistics table

In [43]:
def unique_lemma(file, table_name):
    #Connect to database
    conn = sqlite3.connect(file)
    cursor = conn.cursor()

    cursor.execute(
        f"""
        UPDATE {table_name}
        SET unique_lemmas = (
            SELECT unique_lemmas
            FROM verb_adv_unique_lemmas
            WHERE 
                verb_adv_unique_lemmas.verb = {table_name}.verb
                AND verb_adv_unique_lemmas.verb_compound = {table_name}.verb_compound
    );
    """)

    # Commit and close
    conn.commit()
    conn.close()

In [19]:
unique_lemma(filename, 'verb_adv_log_location')

In [34]:
unique_lemma(filename, 'verb_adv_log_manner')

In [44]:
unique_lemma(filename, 'verb_adv_log_time')

In [54]:
unique_lemma(filename, 'verb_adv_log_amount')

In [63]:
unique_lemma(filename, 'verb_adv_log_state')

#### Add binary logarithm data of unique lemmas to log statistics table

In [45]:
def unique_lemma_log(file, table_name):
    #Connect to database
    conn = sqlite3.connect(file)
    cursor = conn.cursor()

    cursor.execute(
        f"""
        UPDATE {table_name}
        SET log2_unique_lemmas = (
            SELECT log2_unique_lemmas
            FROM verb_adv_unique_lemmas
            WHERE 
                verb_adv_unique_lemmas.verb = {table_name}.verb
                AND verb_adv_unique_lemmas.verb_compound = {table_name}.verb_compound
    );
    """)

    # Commit and close
    conn.commit()
    conn.close()

In [21]:
unique_lemma_log(filename, 'verb_adv_log_location')

In [35]:
unique_lemma_log(filename, 'verb_adv_log_manner')

In [46]:
unique_lemma_log(filename, 'verb_adv_log_time')

In [55]:
unique_lemma_log(filename, 'verb_adv_log_amount')

In [64]:
unique_lemma_log(filename, 'verb_adv_log_state')